In [6]:
dataset='CMF_low.nc'
ds=xr.open_dataset('/projekt1/ag_maahn/data_obs_nobackup/modis/MCD06COSP_D3/merged/'+dataset)
ds

<xarray.Dataset> Size: 414MB
Dimensions:             (time: 8621, longitude: 40, latitude: 30)
Coordinates:
  * time                (time) datetime64[ns] 69kB 2002-07-04 ... 2026-03-11
  * longitude           (longitude) float64 320B -79.5 -78.5 ... -41.5 -40.5
  * latitude            (latitude) float64 240B -19.5 -18.5 -17.5 ... 8.5 9.5
Data variables:
    Mean                (time, longitude, latitude) float64 83MB ...
    Standard_Deviation  (time, longitude, latitude) float64 83MB ...
    Sum                 (time, longitude, latitude) float64 83MB ...
    Pixel_Counts        (time, longitude, latitude) float64 83MB ...
    Sum_Squares         (time, longitude, latitude) float64 83MB ...
Attributes:
    long_name:     Cloud Fraction from Cloud Mask (Low Clouds, CTP GE 680 hPa...
    units:         none
    _FillValue:    -999.0
    valid_min:     0.0
    valid_max:     1.0
    scale_factor:  1.0
    add_offset:    0.0

In [12]:
time_num = (ds.time - ds.time[0]).dt.days.values

def linear_trend(y):
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan
    slope, _, _, p_val, _ = stats.linregress(time_num[mask], y[mask])
    return slope

# Trend für jeden Gitterpunkt berechnen
trend = xr.apply_ufunc(
    linear_trend,
    ds["Mean"],
    input_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float]
)

# Als Dataset speichern
trend_ds = trend.to_dataset(name="trend_Mean")
trend_ds = trend_ds*3650 #trend per decade

In [13]:
trend_ds

<xarray.Dataset> Size: 10kB
Dimensions:     (latitude: 30, longitude: 40)
Coordinates:
  * latitude    (latitude) float64 240B -19.5 -18.5 -17.5 -16.5 ... 7.5 8.5 9.5
  * longitude   (longitude) float64 320B -79.5 -78.5 -77.5 ... -42.5 -41.5 -40.5
Data variables:
    trend_Mean  (longitude, latitude) float64 10kB -0.002897 ... 0.02626